In [155]:
import duckdb
import json
import pandas as pd
import re

In [166]:
d = {}
with open('contracts_abi.json', 'r') as f:
    d = json.loads(f.read())
print(len(d))

1630


In [167]:
l = []
for k, v in d.items():
    v['address'] = k
    abi_func_list = [el['name'] for el in v['abi'] if el['type'] in ['function', 'event']]
    abi_func_list = list(set(abi_func_list))
    words = []
    for el in abi_func_list:
        words += re.split(r'(?<![A-Z])(?=[A-Z])', el)
    words = [x for x in words if x]
    v['abi'] = words
    l.append(v)
print(len(l))

1630


In [168]:
with open('contracts_abi_flat.json', 'w') as f:
    json.dump(l, f)

In [169]:
duckdb.sql("create or replace table t_abi as select * from 'contracts_abi_flat.json'")

In [170]:
duckdb.sql("describe t_abi").show()

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ abi         │ VARCHAR[]   │ YES     │ NULL    │ NULL    │ NULL    │
│ name        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ is_proxy    │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ impl        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ address     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



In [171]:
df = duckdb.sql("select name, abi, address from t_abi where name<>'' limit 20").df()

In [172]:
df.head()

,name,abi,address
0,Swapper,"[is, Authorized, unauthorize, Executed, regist...",0x3c11F6265Ddec22f4d049Dde480615735f451646
1,MainnetSettler,"[execute, Git, Commit, balance, Of]",0x07E594aA718bB872B526e93EEd830a8d2a6A1071
2,TransparentUpgradeableProxy,"[Upgraded, upgrade, To, And, Call, Beacon, Upg...",0xb695F88E6aC201d09269b1e83444dAB290D15395
3,PresaleV1,"[buy, With, ETHWert, buy, With, USDT, incremen...",0xd8B0B0eD366Dd0c3A7DD900CD743502249e7D437
4,GPv2Settlement,"[Order, Invalidated, swap, filled, Amount, Pre...",0x9008D19f58AAbD9eD0D60971565AA8510560ab41


In [173]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   name     20 non-null     object
 1   abi      20 non-null     object
 2   address  20 non-null     object
dtypes: object(3)
memory usage: 608.0+ bytes


In [175]:
for r in df.iterrows():
    func_names = list(r[1]['abi'])
    print(func_names)
    break

['is', 'Authorized', 'unauthorize', 'Executed', 'registry', 'set', 'Smart', 'Vault', 'set', 'Source', 'Unauthorized', 'Paused', 'pause', 'unpause', 'ANY_', 'ADDRESS', 'Smart', 'Vault', 'Set', 'Authorized', 'call', 'smart', 'Vault', 'NAMESPACE', 'get', 'Allowed', 'Sources', 'authorize', 'is', 'Source', 'Allowed', 'paused', 'Source', 'Set', 'Unpaused']
